# Predicting Crash-Test Star Rating

Predicts `OVERALL_STARS` from `equipment_score`, `MODEL_YR`, and `VEHICLE_TYPE`, using a linear regression and a random forest. Given `analysis.ipynb` found only a weak, time-varying correlation between equipment score and star rating (r ranging from -0.21 to +0.27 depending on the model-year window), the honest expectation here is a modest model — this notebook exists to quantify that modesty, not to oversell it.

In [1]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

con = duckdb.connect("../data/nhtsa.duckdb", read_only=True)
df = con.execute("""
    SELECT v.MODEL_YR, v.VEHICLE_TYPE, s.equipment_score, v.OVERALL_STARS
    FROM vehicles v JOIN vehicle_equipment_score s USING (vehicle_id)
    WHERE v.OVERALL_STARS IS NOT NULL AND v.VEHICLE_TYPE IN ('PC', 'MPV', 'TRUCK')
""").df()

X = pd.get_dummies(df[["MODEL_YR", "equipment_score", "VEHICLE_TYPE"]], columns=["VEHICLE_TYPE"])
y = df["OVERALL_STARS"]
print(f"{len(df)} rows, features: {list(X.columns)}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

5434 rows, features: ['MODEL_YR', 'equipment_score', 'VEHICLE_TYPE_MPV', 'VEHICLE_TYPE_PC', 'VEHICLE_TYPE_TRUCK']


## Linear regression

In [2]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

print(f"R^2  = {r2_score(y_test, pred):.3f}")
print(f"MAE  = {mean_absolute_error(y_test, pred):.3f} stars")
print()
for name, coef in zip(X.columns, lr.coef_):
    print(f"  {name:25s} {coef:+.4f}")

R^2  = 0.245
MAE  = 0.391 stars

  MODEL_YR                  +0.0464
  equipment_score           -0.3098
  VEHICLE_TYPE_MPV          +0.0659
  VEHICLE_TYPE_PC           +0.2860
  VEHICLE_TYPE_TRUCK        -0.3519


## Random forest

In [3]:
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print(f"R^2  = {r2_score(y_test, pred):.3f}")
print(f"MAE  = {mean_absolute_error(y_test, pred):.3f} stars")
print()
importances = sorted(zip(X.columns, rf.feature_importances_), key=lambda x: -x[1])
for name, imp in importances:
    print(f"  {name:25s} {imp:.3f}")

R^2  = 0.363
MAE  = 0.341 stars

  MODEL_YR                  0.382
  equipment_score           0.274
  VEHICLE_TYPE_TRUCK        0.268
  VEHICLE_TYPE_PC           0.042
  VEHICLE_TYPE_MPV          0.034


## Conclusion

Both models are modest, as expected: linear regression gets R² = 0.245 (MAE ≈ 0.39 stars), the random forest R² = 0.363 (MAE ≈ 0.34 stars). Neither is strong enough to say "equipment score predicts crash safety" on its own — but the *shape* of the results is informative.

**The linear regression's `equipment_score` coefficient is negative (-0.31)** once `MODEL_YR` and `VEHICLE_TYPE` are held constant — the opposite of what a naive reading of "more equipment = safer" would predict. This isn't a broken model; it's confounding by vehicle segment. Trucks systematically have both lower equipment scores *and* lower star ratings than passenger cars, and once the model has `VEHICLE_TYPE_TRUCK` (coefficient -0.35) to absorb that segment effect, whatever's left over in `equipment_score` picks up a different, weaker, and apparently inverse pattern.

**The random forest's feature importances back this up**: `MODEL_YR` alone accounts for the largest share of predictive power (0.38) — most of what makes a newer vehicle safer is just "it's newer," an industry-wide trend, not its specific equipment list. `equipment_score` (0.27) and `VEHICLE_TYPE_TRUCK` (0.27) contribute roughly equal, smaller shares, which is consistent with them capturing overlapping information (equipment richness tracks vehicle segment) rather than two fully independent signals.

Put together, this matches the hypothesis from `analysis.ipynb`'s sign-flip finding: differences in equipment richness are entangled with *which kind of vehicle* they show up on and *when*, more than they independently drive crash-test outcomes. A cleaner test would model within-segment, within-era variation directly (e.g. a fixed-effects style split by vehicle type and 5-year window) rather than relying on `VEHICLE_TYPE` dummies and `MODEL_YR` as blunt controls — a natural next step if this were extended further.